# 04 — Text-Conditioned Generation

Prepend a text prefix (composer name / style tag) to steer the Music Transformer decoder.
Run `00_setup_and_data.ipynb` first. Needs `MIDI_DIR`, `TRAIN_CSV`, `VAL_CSV`.


In [ ]:
import os, glob
MIDI_DIR   = '/content/maestro'
TRAIN_CSV  = os.path.join(MIDI_DIR, 'train.csv')
VAL_CSV    = os.path.join(MIDI_DIR, 'val.csv')
OUTPUT_DIR = '/content/drive/MyDrive/deep-techno-data/checkpoints/text_conditioned'
VOCAB_PATH = '/content/drive/MyDrive/deep-techno-data/text_vocab.json'

## Build text vocabulary from MAESTRO metadata


In [ ]:
from deepTechno.encoders.text_encoder import build_vocab_from_maestro, TextVocab

maestro_csv = glob.glob(f'{MIDI_DIR}/**/*.csv', recursive=True)[0]
vocab = build_vocab_from_maestro(maestro_csv, save_path=VOCAB_PATH)
print(f'Vocabulary size: {len(vocab)} tokens')

## Composer distribution in vocab


In [ ]:
import pandas as pd, matplotlib.pyplot as plt

meta = pd.read_csv(maestro_csv)
top = meta['canonical_composer'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(10, 4))
top.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Pieces')
ax.set_title('Top 15 composers (text prefix candidates)')
plt.tight_layout(); plt.show()

## Text encoder demo


In [ ]:
import torch
from deepTechno.encoders.text_encoder import TextEncoder

d_model  = 512
encoder  = TextEncoder(vocab_size=len(vocab), d_model=d_model, max_text_len=32)

text   = 'Chopin'
ids    = vocab.encode(text)
t      = torch.tensor(ids).unsqueeze(0)
prefix = encoder(t)
print(f'Input: "{text}" → {len(ids)} tokens')
print(f'Prefix shape: {prefix.shape}  (B, T_text, d_model={d_model})')

## Training a text-conditioned transformer

The simplest integration: prepend the text prefix embeddings to the MIDI token embeddings, then feed the combined sequence to MusicTransformer as usual. The transformer attends over both text and MIDI tokens causally.


In [ ]:
# Prototype forward pass (not a full training loop — demonstrates the prefix approach)
from deepTechno.model.transformer import MusicTransformer
from deepTechno.model.constants import VOCAB_SIZE

midi_model    = MusicTransformer(n_layers=6, num_heads=8, d_model=d_model, rpr=True)
text_encoder  = TextEncoder(vocab_size=len(vocab), d_model=d_model)

# Fake batch
B         = 2
T_text    = 8
T_midi    = 64
text_ids  = torch.randint(0, len(vocab), (B, T_text))
midi_ids  = torch.randint(0, VOCAB_SIZE, (B, T_midi))

text_prefix   = text_encoder(text_ids)                      # (B, T_text, d_model)
midi_embeds   = midi_model.embedding(midi_ids)              # (B, T_midi, d_model)
combined      = torch.cat([text_prefix, midi_embeds], dim=1)  # (B, T_text+T_midi, d_model)
print('Combined embedding shape:', combined.shape)
print('This combined sequence would be passed to the transformer decoder.')

## Generate conditioned on a composer name


In [ ]:
# Full conditioned generation requires a trained text-conditioned model.
# Below shows how to prepare the prefix for inference once you have one.

composer = 'Beethoven'
prefix_ids = vocab.encode(composer, max_len=16)
print(f'Composer: "{composer}"')
print(f'Token ids: {prefix_ids}')
print(f'Tokens: {[vocab.idx2token[i] for i in prefix_ids]}')
print('\nPrepend these ids to your MIDI primer and call generate_from_primer().')